LOGIN PAGE


In [1]:
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 64.7 MB/s eta 0:00:00


In [2]:
%%writefile app.py
import os, sqlite3, jwt, bcrypt, datetime, time, streamlit as st
import plotly.graph_objects as go
from streamlit_option_menu import option_menu

# 🚀 Force Streamlit Light Theme
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#ffd803"\nbackgroundColor="#f9fcfc"\nsecondaryBackgroundColor="#e3f6f5"\ntextColor="#2d334a"\n')

st.set_page_config(page_title="Infosys Portal", page_icon="⚡", layout="wide", initial_sidebar_state="expanded")

COLORS = {
    "bg_main": "#f9fcfc", "bg_sidebar": "#e3f6f5", "bg_card": "#ffffff", "bg_card_alt": "#bae8e8",
    "text_main": "#2d334a", "text_heading": "#272343", "text_muted": "#64748b",
    "accent": "#ffd803", "accent_hover": "#e6c300", "accent_text": "#272343",
    "border": "#272343", "border_light": "#bae8e8", "success": "#34d399", "danger": "#f87171"
}

JWT_SECRET = "super-secret-infosys-key-2026"

# 🚀 Neo-Brutalist CSS + Visible Sidebar Re-Open Button
st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&family=Inter:wght@300;400;500;600&display=swap');
    html, body, .stApp {{ background: {COLORS['bg_main']} !important; font-family: 'Inter', sans-serif !important; color: {COLORS['text_main']} !important; }}

    /* 🚀 DO NOT HIDE HEADER! Only hide footer and bottom decoration so toggle arrow stays visible! */
    footer, div[data-testid="stDecoration"] {{ visibility: hidden !important; display: none !important; }}
    header {{ background: transparent !important; z-index: 999999 !important; }}

    /* 🚀 Highlight Streamlit's Native Sidebar Re-Open Button (Top-Left) as a Yellow Badge! */
    button[kind="header"], div[data-testid="stSidebarCollapsedControl"] button {{
        visibility: visible !important; display: flex !important; opacity: 1 !important;
        background-color: {COLORS['accent']} !important; border: 2px solid {COLORS['border']} !important;
        border-radius: 8px !important; padding: 6px !important; margin: 8px !important;
        box-shadow: 3px 3px 0px {COLORS['border']} !important;
    }}
    button[kind="header"] svg, div[data-testid="stSidebarCollapsedControl"] svg {{
        fill: {COLORS['text_heading']} !important; color: {COLORS['text_heading']} !important; stroke: {COLORS['text_heading']} !important;
    }}

    .block-container {{ padding: 2rem 2.5rem !important; max-width: 1200px; }}
    h1, h2, h3, h4 {{ font-family: 'Poppins', sans-serif !important; color: {COLORS['text_heading']} !important; }}
    label p {{ font-weight: 600 !important; color: {COLORS['text_heading']} !important; }}

    /* Inputs */
    div[data-baseweb="base-input"], div[data-baseweb="select"] > div {{ background-color: transparent !important; border: none !important; }}
    div[data-baseweb="input"], div[data-baseweb="select"] {{ background-color: {COLORS['bg_card']} !important; border: 2px solid {COLORS['border']} !important; border-radius: 10px !important; }}
    div[data-baseweb="input"]:focus-within {{ border-color: {COLORS['accent']} !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; }}
    input, div[data-baseweb="select"] span {{ color: {COLORS['text_main']} !important; -webkit-text-fill-color: {COLORS['text_main']} !important; }}

    /* Buttons */
    div[data-testid="stButton"] button {{
        background-color: {COLORS['accent']} !important; color: {COLORS['accent_text']} !important;
        border: 2px solid {COLORS['border']} !important; border-radius: 10px !important;
        font-family: 'Inter', sans-serif !important; font-weight: 700 !important; font-size: 14px !important;
        height: 48px !important; min-height: 48px !important; white-space: nowrap !important;
        display: flex !important; align-items: center !important; justify-content: center !important;
        padding: 0px 16px !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; width: 100%; transition: all 0.2s ease !important;
    }}
    div[data-testid="stButton"] button:hover {{
        background-color: {COLORS['accent_hover']} !important; transform: translate(-2px, -2px) !important;
        box-shadow: 6px 6px 0px {COLORS['border']} !important;
    }}
    section[data-testid="stSidebar"] {{ background: {COLORS['bg_sidebar']} !important; border-right: 2px solid {COLORS['border']} !important; }}
    .pn-card {{ background: {COLORS['bg_card']}; border: 2px solid {COLORS['border']}; border-radius: 14px; padding: 24px; box-shadow: 4px 4px 0px {COLORS['border_light']}; }}
</style>
""", unsafe_allow_html=True)

def get_db(): return sqlite3.connect("infosys_portal.db", check_same_thread=False)
def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False

with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, email TEXT UNIQUE,
        password_hash TEXT, security_question TEXT, security_answer_hash TEXT)""")
    if not conn.execute("SELECT id FROM users WHERE email='infosys@ai'").fetchone():
        conn.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)",
                     ("Administrator", "infosys@ai", hash_txt("admin@123"), "What is your pet name?", hash_txt("admin")))

def make_jwt(email): return jwt.encode({"email": email, "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)}, JWT_SECRET, algorithm="HS256")
def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except: return None

for k, v in [("token", None), ("page", "Login"), ("reset_email", None), ("reset_mode", None)]:
    if k not in st.session_state: st.session_state[k] = v

def navigate(p): st.session_state.page = p; st.rerun()

def auth_header(title, sub="Intelligent Analytics Portal"):
    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 1rem;">
        <div style="font-size:40px;margin-bottom:10px;">⚡</div>
        <h1 style="font-size:2rem !important;margin:0;">Infosys Portal</h1>
        <p style="color:{COLORS['text_muted']};font-size:14px;margin:4px 0 0;">{sub}</p>
    </div>
    <div style="text-align:center;margin-bottom:1.5rem;"><span style="font-size:1.1rem;font-weight:700;color:{COLORS['text_heading']};">{title}</span></div>
    """, unsafe_allow_html=True)

# ============================================================
# PAGE ROUTING
# ============================================================
if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.45, 1])
    with mid:
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="••••••••")
            st.markdown("<br>", unsafe_allow_html=True)

            col_l, col_c, col_r = st.columns([1, 1.15, 1.3])
            if col_l.button("Sign In →", use_container_width=True):
                with get_db() as c: r = c.execute("SELECT password_hash FROM users WHERE email=?", (email,)).fetchone()
                if r and check_txt(pwd, r[0]): st.session_state.token = make_jwt(email); navigate("Dashboard")
                else: st.error("❌ Invalid credentials.")
            if col_c.button("Create Account", use_container_width=True): navigate("Signup")
            if col_r.button("Forgot Password", use_container_width=True): navigate("Forgot")

        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Join Infosys Portal today")
            uname = st.text_input("Full name / Username", placeholder="Jane Doe")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="Min. 8 characters")
            confirm_pwd = st.text_input("Confirm password", type="password", placeholder="Re-enter password")
            sq = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favourite city?"])
            sa = st.text_input("Your answer", placeholder="Security answer")
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Create Account & Login →", use_container_width=True):
                if not uname or not email or len(pwd) < 8 or not sa:
                    st.error("⚠️ Please fill all fields (password min 8 chars).")
                elif pwd != confirm_pwd:
                    st.error("❌ Passwords do not match.")
                else:
                    try:
                        with get_db() as c:
                            c.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)", (uname, email, hash_txt(pwd), sq, hash_txt(sa.lower().strip())))
                        st.session_state.token = make_jwt(email)
                        st.success("✅ Account created!")
                        time.sleep(1)
                        navigate("Dashboard")
                    except sqlite3.IntegrityError:
                        st.error("❌ Email or Username already registered.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Back to Sign In", use_container_width=True): navigate("Login")

        elif st.session_state.page == "Forgot":
            auth_header("Reset your password", "Choose your verification method")
            if not st.session_state.reset_email:
                email = st.text_input("Registered email address", placeholder="you@infosys.com").lower().strip()
                st.markdown("<br>", unsafe_allow_html=True)

                col_sq, col_otp = st.columns(2)
                if col_sq.button("Via Security Question", use_container_width=True):
                    with get_db() as c: r = c.execute("SELECT security_question FROM users WHERE email=?", (email,)).fetchone()
                    if r:
                        st.session_state.reset_email = email
                        st.session_state.sq_p = r[0]
                        st.session_state.reset_mode = "sq"
                        st.rerun()
                    else: st.error("❌ Email not found.")

                if col_otp.button("Via OTP", use_container_width=True):
                    pass
            else:
                if st.session_state.get("reset_mode") == "sq":
                    st.info(f"❓ **Security Question:** {st.session_state.sq_p}")
                    ans = st.text_input("Your answer").lower().strip()
                    npw = st.text_input("New password (min 8 chars)", type="password")
                    confirm_npw = st.text_input("Confirm new password", type="password")
                    st.markdown("<br>", unsafe_allow_html=True)
                    if st.button("Reset Password →", use_container_width=True):
                        if len(npw) < 8:
                            st.error("⚠️ Password must be at least 8 characters long.")
                        elif npw != confirm_npw:
                            st.error("❌ Passwords do not match.")
                        else:
                            with get_db() as c: r = c.execute("SELECT security_answer_hash FROM users WHERE email=?", (st.session_state.reset_email,)).fetchone()
                            if r and check_txt(ans, r[0]):
                                with get_db() as c: c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                                st.success("✅ Password updated successfully!"); time.sleep(1); st.session_state.reset_email = None; navigate("Login")
                            else: st.error("❌ Incorrect security answer.")
            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Cancel", use_container_width=True):
                st.session_state.reset_email = None
                st.session_state.reset_mode = None
                navigate("Login")

# ============================================================
# DASHBOARDS (ADMIN vs USER)
# ============================================================
else:
    payload = verify_jwt(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    email = payload["email"]
    with get_db() as c: uname = c.execute("SELECT username FROM users WHERE email=?", (email,)).fetchone()[0]

    # 🚀 SIDEBAR MENU
    with st.sidebar:
        st.markdown(f"""
        <div style="padding:16px 8px;text-align:center;">
            <div style="font-size:28px;">⚡</div>
            <div style="font-weight:700;font-size:16px;color:{COLORS['text_heading']};">Infosys Portal</div>
            <div style="font-size:11px;color:{COLORS['text_muted']};">{"Admin Panel" if email=="infosys@ai" else "Intelligent Analytics"}</div>
        </div><hr style="border-color:{COLORS['border_light']};">
        """, unsafe_allow_html=True)

        opts = ["Dashboard", "Settings", "Logout"] if email=="infosys@ai" else ["Dashboard", "Analytics", "Reports", "Logout"]
        menu = option_menu(None, opts, icons=["house", "gear", "box-arrow-right"] if email=="infosys@ai" else ["house", "graph-up", "file-text", "box-arrow-right"],
                           styles={"container": {"background-color": COLORS['bg_sidebar']}, "nav-link-selected": {"background-color": COLORS['accent'], "color": COLORS['accent_text']}})
        if menu == "Logout":
            st.session_state.token = None
            st.session_state.page = "Login"
            st.rerun()

    # 🚀 ADMIN DASHBOARD (infosys@ai)
    if email == "infosys@ai":
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Infosys Portal</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Admin Control Panel</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">🛡️ {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown(f"""
        <div class="pn-card" style="text-align:center;padding:60px 20px;">
            <h1 style="font-size:36px !important;margin-bottom:10px;">🛡️ Admin Dashboard</h1>
            <p style="color:{COLORS['text_muted']};font-size:16px;font-weight:500;">Welcome to the Administrator area.</p>
        </div>
        """, unsafe_allow_html=True)

    # 🚀 REGULAR USER DASHBOARD
    else:
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Infosys Portal</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Analytics Dashboard</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">👤 {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        c1, c2, c3, c4 = st.columns(4)
        for col, icon, lbl, val in [(c1, "📄", "Documents Indexed", "128"), (c2, "🔍", "Searches Today", "47"),
                                    (c3, "📊", "Efficiency Score", "98.4%"), (c4, "🛡️", "Security Status", "Secured")]:
            col.markdown(f"""
            <div class="pn-card" style="text-align:center;">
                <div style="font-size:28px;">{icon}</div>
                <div style="font-size:26px;font-weight:700;color:{COLORS['text_heading']};">{val}</div>
                <div style="color:{COLORS['text_muted']};font-size:12px;font-weight:600;">{lbl}</div>
            </div>
            """, unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        fig = go.Figure(go.Indicator(mode="gauge+number", value=92, title={"text": "System Health Index", "font": {"color": COLORS['text_heading'], "size": 14}},
                        gauge={"axis": {"range": [0, 100]}, "bar": {"color": COLORS['accent']}, "bgcolor": COLORS['bg_card_alt'], "borderwidth": 1, "bordercolor": COLORS['border']}))
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font={"color": COLORS['text_main'], "family": "Inter"}, height=260, margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)


Writing app.py


In [9]:
import os
import time
import subprocess
from pyngrok import ngrok
from google.colab import userdata

# 1. Retrieve your secret token securely from Colab Secrets
NGROK_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Kill any existing ngrok tunnels or streamlit sessions
ngrok.kill()
!pkill -f streamlit

# 3. Start Streamlit in the background on port 8501
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 4. Open ngrok tunnel
public_url = ngrok.connect(8501).public_url
print("=" * 60)
print(f"🚀 Infosys Portal Live URL: {public_url}")
print("=" * 60)
print("⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.")

try:
    # Keep the cell active so Ctrl+C can be intercepted
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n" + "🛑" * 30)
    print("Received Ctrl+C / Stop signal. Shutting down...")
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("✅ Ngrok tunnel closed and Streamlit server stopped gracefully.")


🚀 Infosys Portal Live URL: https://intermeningeal-ernesto-piliferous.ngrok-free.dev
⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.

🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑
Received Ctrl+C / Stop signal. Shutting down...
✅ Ngrok tunnel closed and Streamlit server stopped gracefully.


In [4]:
!pip install streamlit pyjwt bcrypt python-dotenv pyngrok -q

In [13]:
%%writefile db.py
import sqlite3
import bcrypt
import datetime
import time

DB_NAME = "users.db"
max_login_attempts = 3
lockout_time = 300  # 5 minutes in seconds

def init_db():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    # Users table
    c.execute('''CREATE TABLE IF NOT EXISTS users
                 (email TEXT PRIMARY KEY, password BLOB, created_at TEXT)''')

    # Password History table
    c.execute('''CREATE TABLE IF NOT EXISTS password_history
                 (id INTEGER PRIMARY KEY AUTOINCREMENT,
                  email TEXT,
                  password BLOB,
                  set_at TEXT,
                  FOREIGN KEY(email) REFERENCES users(email))''')

    # Login Attempts table
    c.execute('''CREATE TABLE IF NOT EXISTS login_attempts
                 (email TEXT PRIMARY KEY, attempts INTEGER, last_attempt REAL)''')

    conn.commit()
    conn.close()

    # Ensure Admin Exists
    init_admin()

def init_admin():
    if not check_user_exists("admin@llm.com"):
        register_user("admin@llm.com", "Admin@123")
        print("Admin account created.")

def _get_timestamp():
    return datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

# --- Rate Limiting ---
def get_login_attempts(Email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT attempts, last_attempt FROM login_attempts WHERE email = ?", (email,))
    data = c.fetchone()
    conn.close()
    return data if data else (0, 0)

def increment_login_attempts(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    attempts, _ = get_login_attempts(email)
    new_attempts = attempts + 1
    now = time.time()

    c.execute("INSERT OR REPLACE INTO login_attempts (email, attempts, last_attempt) VALUES (?, ?, ?)",
              (email, new_attempts, now))
    conn.commit()
    conn.close()

def reset_login_attempts(Email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("DELETE FROM login_attempts WHERE email = ?", (email,))
    conn.commit()
    conn.close()

def is_rate_limited(Email):
    attempts, last_attempt = get_login_attempts(Email)
    if attempts >= max_login_attempts:
        if time.time() - last_attempt < lockout_time:
            return True, lockout_time - (time.time() - last_attempt)
        else:
            # Lockout expired
            reset_login_attempts(Email)
    return False, 0

# --- User Management ---
def register_user(email, password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    try:
        salt = bcrypt.gensalt()
        hashed = bcrypt.hashpw(password.encode('utf-8'), salt)
        now = _get_timestamp()

        c.execute("INSERT INTO users (email, password, created_at) VALUES (?, ?, ?)", (email, hashed, now))
        c.execute("INSERT INTO password_history (email, password, set_at) VALUES (?, ?, ?)", (email, hashed, now))

        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

def authenticate_user(Email, password):
    # Check rate limit first in app.py or here? Better to check before auth logic.
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT password FROM users WHERE email = ?", (email,))
    data = c.fetchone()
    conn.close()
    if data:
        stored_hash = data[0]
        if bcrypt.checkpw(password.encode('utf-8'), stored_hash):
            reset_login_attempts(email) # Success reset
            return True

    # Failed
    increment_login_attempts(email)
    return False

def check_is_old_password(email, password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT password, set_at FROM password_history WHERE email = ? ORDER BY set_at DESC", (email,))
    history = c.fetchall()
    conn.close()

    for stored_hash, set_at in history:
        if bcrypt.checkpw(password.encode('utf-8'), stored_hash):
            return set_at
    return None

def check_password_reused(Email, new_password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT password FROM password_history WHERE email = ?", (email,))
    history = c.fetchall()
    conn.close()

    for (stored_hash,) in history:
        if bcrypt.checkpw(new_password.encode('utf-8'), stored_hash):
            return True
    return False

def check_user_exists(Email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT 1 FROM users WHERE email = ?", (Email,))
    data = c.fetchone()
    conn.close()
    return data is not None

def update_password(Email, new_password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    salt = bcrypt.gensalt()
    hashed = bcrypt.hashpw(new_password.encode('utf-8'), salt)
    now = _get_timestamp()

    c.execute("UPDATE users SET password = ? WHERE email = ?", (hashed, Email))
    c.execute("INSERT INTO password_history (email, password, set_at) VALUES (?, ?, ?)", (email, hashed, now))

    conn.commit()
    conn.close()

# --- Admin Functions ---
def get_all_users():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT email, created_at FROM users")
    data = c.fetchall()
    conn.close()
    return data

def delete_user(Email):
    if email == "admin@llm.com": return False # Protect admin
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("DELETE FROM users WHERE email = ?", (email,))
    c.execute("DELETE FROM password_history WHERE email = ?", (email,))
    c.execute("DELETE FROM login_attempts WHERE email = ?", (email,))
    conn.commit()
    conn.close()
    return True


Overwriting db.py


In [14]:
%%writefile app.py
import os, sqlite3, jwt, bcrypt, datetime, time, secrets, smtplib, streamlit as st
from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# 🚀 1. Force Streamlit Light Theme
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#ffd803"\nbackgroundColor="#f9fcfc"\nsecondaryBackgroundColor="#e3f6f5"\ntextColor="#2d334a"\n')

st.set_page_config(page_title="OTP Password Reset", page_icon="🔐", layout="wide")

COLORS = {
    "bg_main": "#f9fcfc", "bg_card": "#ffffff", "bg_card_alt": "#bae8e8",
    "text_main": "#2d334a", "text_heading": "#272343", "text_muted": "#64748b",
    "accent": "#ffd803", "accent_hover": "#e6c300", "accent_text": "#272343",
    "border": "#272343", "border_light": "#bae8e8"
}

JWT_SECRET = os.getenv("JWT_SECRET", "super-secret-otp-key-2026")
SENDER_EMAIL = "mohamedsipli@gmail.com"  # 🚀 Your sender email address
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")  # 🚀 Pulled securely from Colab Secrets!
OTP_EXPIRY_MINUTES = 5

# 🚀 2. Apply Exact Light Neo-Brutalist CSS + Clean Card Container
st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&family=Inter:wght@300;400;500;600&display=swap');
    html, body, .stApp {{ background: {COLORS['bg_main']} !important; font-family: 'Inter', sans-serif !important; color: {COLORS['text_main']} !important; }}
    header, footer, #MainMenu {{ visibility: hidden !important; display: none !important; }}
    .block-container {{ padding: 3rem 2rem !important; max-width: 800px; }}
    h1, h2, h3, h4 {{ font-family: 'Poppins', sans-serif !important; color: {COLORS['text_heading']} !important; }}
    label p {{ font-weight: 600 !important; color: {COLORS['text_heading']} !important; }}

    /* 🚀 Style Streamlit's Native Container to act as our Neo-Brutalist Card */
    div[data-testid="stVerticalBlockBorderWrapper"] {{
        background-color: {COLORS['bg_card']} !important; border: 2px solid {COLORS['border']} !important;
        border-radius: 16px !important; padding: 24px !important; box-shadow: 6px 6px 0px {COLORS['border_light']} !important;
    }}

    /* Inputs */
    div[data-baseweb="base-input"] {{ background-color: transparent !important; border: none !important; }}
    div[data-baseweb="input"] {{ background-color: {COLORS['bg_card']} !important; border: 2px solid {COLORS['border']} !important; border-radius: 10px !important; }}
    div[data-baseweb="input"]:focus-within {{ border-color: {COLORS['accent']} !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; }}
    input {{ color: {COLORS['text_main']} !important; -webkit-text-fill-color: {COLORS['text_main']} !important; font-size: 16px !important; }}

    /* Buttons */
    div[data-testid="stButton"] button {{
        background-color: {COLORS['accent']} !important; color: {COLORS['accent_text']} !important;
        border: 2px solid {COLORS['border']} !important; border-radius: 10px !important;
        font-family: 'Inter', sans-serif !important; font-weight: 700 !important; font-size: 15px !important;
        height: 48px !important; min-height: 48px !important; white-space: nowrap !important;
        display: flex !important; align-items: center !important; justify-content: center !important;
        box-shadow: 4px 4px 0px {COLORS['border']} !important; width: 100%; transition: all 0.2s ease !important;
    }}
    div[data-testid="stButton"] button:hover {{
        background-color: {COLORS['accent_hover']} !important; transform: translate(-2px, -2px) !important;
        box-shadow: 6px 6px 0px {COLORS['border']} !important;
    }}
</style>
""", unsafe_allow_html=True)

# ── 3. Database Initialization & Pre-Seeding ──
def get_db(): return sqlite3.connect("users_otp.db", check_same_thread=False)
def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False

with get_db() as conn:
    conn.execute("CREATE TABLE IF NOT EXISTS users (email TEXT PRIMARY KEY, password_hash TEXT)")
    for email in ["springboardmentor018@gmail.com", "springboardmentor038@gmail.com"]:
        if not conn.execute("SELECT email FROM users WHERE email=?", (email,)).fetchone():
            conn.execute("INSERT INTO users VALUES (?, ?)", (email, hash_txt("Welcome@123")))

# ── 4. Helper Functions (Secrets OTP & JWT Token) ──
def generate_otp(): return f"{secrets.randbelow(900000) + 100000}"

def make_otp_token(email, otp):
    payload = {"sub": email, "otp_hash": hash_txt(otp), "type": "password_reset_otp", "iat": datetime.datetime.utcnow(), "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)}
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_otp_token(token, input_otp, email):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
        if payload.get("sub") != email or payload.get("type") != "password_reset_otp": return False, "Security token mismatch."
        if check_txt(input_otp, payload["otp_hash"]): return True, "Valid"
        return False, "Invalid 6-digit OTP code."
    except jwt.ExpiredSignatureError: return False, f"⚠️ This OTP code expired after {OTP_EXPIRY_MINUTES} minutes. Please request a new one."
    except Exception as e: return False, "Invalid or corrupted verification token."

def send_professional_email(to_email, otp, app_pass):
    # 🚀 Anti-Spam Setup: Use MIMEMultipart('alternative') and standard RFC headers
    msg = MIMEMultipart('alternative')
    msg['From'] = f"Infosys Support <{SENDER_EMAIL}>"
    msg['To'] = to_email
    msg['Subject'] = "Infosys Portal - Verification Code"
    msg['Date'] = formatdate(localtime=True)  # Adds RFC compliant Date header
    msg['Message-ID'] = make_msgid()          # Prevents spam filtering
    msg['Reply-To'] = SENDER_EMAIL

    # Plain text version (Essential to bypass Gmail spam filters!)
    text_body = f"Your verification code for Infosys Portal is: {otp}\nThis code will expire in {OTP_EXPIRY_MINUTES} minutes.\nIf you did not request this code, please ignore this email."

    # Clean HTML version without aggressive spam-trigger formatting
    html_body = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <style>
            body {{ font-family: Arial, sans-serif; background-color: #f9fcfc; margin: 0; padding: 20px; }}
            .container {{ max-width: 500px; margin: 0 auto; background-color: #ffffff; border: 2px solid #272343; border-radius: 12px; padding: 30px; text-align: center; }}
            .title {{ color: #272343; font-size: 20px; font-weight: bold; margin-bottom: 15px; }}
            .text {{ color: #4a5568; font-size: 15px; line-height: 1.5; margin-bottom: 20px; }}
            .otp-box {{ background-color: #ffd803; color: #272343; font-size: 28px; font-weight: bold; letter-spacing: 5px; padding: 15px 20px; border: 2px solid #272343; border-radius: 8px; display: inline-block; margin: 10px 0; }}
            .footer {{ color: #718096; font-size: 12px; margin-top: 25px; border-top: 1px solid #edf2f7; padding-top: 15px; }}
        </style>
    </head>
    <body>
        <div class="container">
            <div class="title">Infosys Portal Verification</div>
            <div class="text">We received a request to reset your password for <b>{to_email}</b>. Please use the verification code below:</div>
            <div class="otp-box">{otp}</div>
            <div class="text">This code expires in <b>{OTP_EXPIRY_MINUTES} minutes</b>.</div>
            <div class="footer">If you did not request this code, you can safely ignore this email.<br>&copy; 2026 Infosys Portal.</div>
        </div>
    </body>
    </html>
    """
    msg.attach(MIMEText(text_body, 'plain'))
    msg.attach(MIMEText(html_body, 'html'))
    try:
        s = smtplib.SMTP('smtp.gmail.com', 587)
        s.starttls()
        s.login(SENDER_EMAIL, app_pass if app_pass else "")
        s.sendmail(SENDER_EMAIL, to_email, msg.as_string())
        s.quit()
        return True, "Email sent successfully!"
    except Exception as e: return False, f"SMTP Error: {str(e)}"

# ── 5. Session State Management ──
for k, v in [("stage", "email"), ("reset_email", ""), ("jwt_token", "")]:
    if k not in st.session_state: st.session_state[k] = v

def restart_flow(): st.session_state.stage = "email"; st.session_state.reset_email = ""; st.rerun()

# ── 6. Single Page OTP UI ──
st.markdown("""
<div style="text-align:center;padding:1rem 0 1.5rem;">
    <div style="font-size:48px;margin-bottom:10px;">🔐</div>
    <h1 style="font-size:2.2rem !important;margin:0;">Secure Password Reset</h1>
    <p style="color:#64748b;font-size:15px;margin:6px 0 0;">Verify your identity with a 6-digit OTP code (5-minute expiry)</p>
</div>
""", unsafe_allow_html=True)

_, col_center, _ = st.columns([1, 2, 1])
with col_center:
    # 🚀 Native Streamlit Bordered Container = Perfect Card without empty top boxes!
    with st.container(border=True):

        # ── STAGE 1: ASK EMAIL ──
        if st.session_state.stage == "email":
            st.markdown("<h3 style='margin-top:0;font-size:18px;text-align:center;'>Enter your email ID</h3>", unsafe_allow_html=True)
            # 🚀 Changed placeholder to "Enter your email ID"
            email = st.text_input("Email Address", placeholder="Enter your email ID", label_visibility="collapsed").lower().strip()
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Send Verification Code →", use_container_width=True):
                if not email: st.error("⚠️ Please enter an email address.")
                elif not EMAIL_PASSWORD:
                    st.error("❌ Gmail App Password not found! Please add `EMAIL_PASSWORD` to your Colab Secrets tab.")
                else:
                    with get_db() as c: r = c.execute("SELECT 1 FROM users WHERE email=?", (email,)).fetchone()
                    if r:
                        otp = generate_otp()
                        with st.spinner(f"Sending 6-digit OTP (expires in {OTP_EXPIRY_MINUTES} mins)..."):
                            ok, msg = send_professional_email(email, otp, EMAIL_PASSWORD)
                        if ok:
                            st.session_state.reset_email = email
                            st.session_state.jwt_token = make_otp_token(email, otp)
                            st.session_state.stage = "otp"
                            st.success("✅ 6-digit OTP sent! Check your inbox."); time.sleep(1); st.rerun()
                        else: st.error(f"❌ {msg}")
                    else: st.error("❌ Email address not registered in the system.")

        # ── STAGE 2: ENTER OTP ──
        elif st.session_state.stage == "otp":
            st.markdown("<h3 style='margin-top:0;font-size:18px;text-align:center;'>Enter 6-Digit OTP Code</h3>", unsafe_allow_html=True)
            st.info(f"📧 Code sent to **{st.session_state.reset_email}** (Valid for {OTP_EXPIRY_MINUTES} mins).")
            otp_input = st.text_input("6-Digit Verification Code", placeholder="e.g. 849201", max_chars=6, label_visibility="collapsed")
            st.markdown("<br>", unsafe_allow_html=True)

            c1, c2 = st.columns([1.2, 1])
            if c1.button("Verify Code →", use_container_width=True):
                if not otp_input or len(otp_input) != 6: st.error("⚠️ Please enter the valid 6-digit code.")
                else:
                    ok, msg = verify_otp_token(st.session_state.jwt_token, otp_input, st.session_state.reset_email)
                    if ok:
                        st.session_state.stage = "reset"
                        st.success("✅ Code verified successfully!"); time.sleep(1); st.rerun()
                    else: st.error(f"❌ {msg}")
            if c2.button("← Back", use_container_width=True): restart_flow()

        # ── STAGE 3: NEW PASSWORD ──
        elif st.session_state.stage == "reset":
            st.markdown("<h3 style='margin-top:0;font-size:18px;text-align:center;'>Create New Password</h3>", unsafe_allow_html=True)
            npw = st.text_input("New Password (min 8 chars)", type="password", placeholder="New Password (min 8 chars)")
            confirm_npw = st.text_input("Confirm New Password", type="password", placeholder="Confirm New Password")
            st.markdown("<br>", unsafe_allow_html=True)

            c1, c2 = st.columns([1.2, 1])
            if c1.button("Update Password →", use_container_width=True):
                if len(npw) < 8: st.error("⚠️ Password must be at least 8 characters long.")
                elif npw != confirm_npw: st.error("❌ Passwords do not match.")
                else:
                    with get_db() as c: c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                    st.success("🎉 Password updated successfully! Your account is secure.")
                    time.sleep(2)
                    restart_flow()
            if c2.button("← Cancel", use_container_width=True): restart_flow()


Overwriting app.py


In [ ]:
import os, time, subprocess
from pyngrok import ngrok
from google.colab import userdata

# 1. Safely retrieve secrets from Colab
try:
    os.environ['Email'] = userdata.get('Email')
    os.environ['JWT_SECRET'] = "super-secure-otp-secret-key-2026"
    ngrok_token = userdata.get('NGROK_AUTH_TOKEN')
    if ngrok_token: ngrok.set_auth_token(ngrok_token)
except Exception as e:
    print(f"⚠️ Secret Error: Make sure EMAIL_PASSWORD and NGROK_AUTHTOKEN exist in Colab Secrets.")

# 🚀 2. Kill ONLY old Streamlit servers (Do NOT kill Ngrok so we don't lock your domain!)
!pkill -f streamlit
time.sleep(2)

# 3. Start Streamlit server cleanly in the background with new UI
process = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501'], env=os.environ.copy(), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# 🚀 4. Safely Reuse or Connect Tunnel
try:
    tunnels = ngrok.get_tunnels()
    if tunnels:
        public_url = tunnels[0].public_url
        print("♻️ Reusing existing active Ngrok tunnel...")
    else:
        public_url = ngrok.connect(8501).public_url
        print("🚀 Created new Ngrok tunnel...")

    print("=" * 65)
    print(f"👉 OTP Password Reset Portal Live URL: {public_url}")
    print("=" * 65)
    print("⏳ Server running! Press [Ctrl + C] or click the Colab Stop button to shut down.")
    while True: time.sleep(1)

except Exception as e:
    print(f"⚠️ Notice: {e}")
    print("🔄 Waiting 5 seconds for Ngrok cloud servers to release domain lock...")
    ngrok.kill()
    !pkill -f ngrok
    time.sleep(5)  # 🚀 Give Ngrok cloud servers time to release your static endpoint!
    public_url = ngrok.connect(8501).public_url
    print("=" * 65)
    print(f"👉 OTP Password Reset Portal Live URL: {public_url}")
    print("=" * 65)
    while True: time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Shutting down server...")
    !pkill -f streamlit
    print("✅ Streamlit stopped. (Ngrok kept alive for faster re-runs)")


🚀 Created new Ngrok tunnel...
👉 OTP Password Reset Portal Live URL: https://intermeningeal-ernesto-piliferous.ngrok-free.dev
⏳ Server running! Press [Ctrl + C] or click the Colab Stop button to shut down.
